# 13.4 為什麼要有參考模型？


你想讓新模型比原本更偏愛簡潔正確的回答，需要一把固定尺。若尺也跟著每次更新伸縮，就難以分清改了多少。SFT（supervised fine-tuning，監督式微調）先用理想助手回答教基礎能力；DPO（direct preference optimization，直接偏好最佳化）再用同題兩篇回答的偏好，調整相對原模型的回答傾向。這種偏好訓練保留一份原來對話模型，叫reference參考模型；正在更新的那份叫policy策略模型，它負責之後實際回答。

前置是[13.3整段log分數](https://birdhackor.github.io/tiny-perceptron-vlm/13.3.html)與[11.2凍結](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)。先算policy的「較佳分數減較差分數」，再減掉reference原本的同一差。這叫相對margin差距。例子中參考為-4與-3，差-1；更新後策略為-3與-3，差0，相對進步就是0−(-1)=1。我們看的是偏好相對原基準增強多少。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import copy
import torch
from tiny_perceptron.model import TinyLM, ModelConfig

policy = TinyLM(ModelConfig(width=8))
reference = copy.deepcopy(policy).eval().requires_grad_(False)
before = reference.embedding.weight.clone()
with torch.no_grad():
    policy.embedding.weight.add_(0.1)
scores = reference(torch.tensor([[1, 2]]))["logits"]
print("參考未跟著改", torch.equal(before, reference.embedding.weight))
print("參考不記梯度", scores.requires_grad)

輸入是一個小隨機模型，用來檢查複製與凍結機制，不是已訓好的偏好基模。`deepcopy`複製整份獨立權重，不能只是`reference=policy`，那會指向同一對象。`requires_grad_(False)`凍結參考參數，`eval()`設評估行為，`no_grad`只包住人為加0.1的原地修改，避免把這個測試動作記成待求導計算。參考分數的計算放在區塊外，讓一般梯度記錄仍開著；此時輸入是整數ID、不求梯度，輸出是否需要梯度就能核對參考參數的凍結狀態。

我們人為將policy的embedding每項加0.1，輸出第一行True：參考沒有跟著變；第二行False：即使計算時沒有用`no_grad`關閉記錄，已凍結的參考分數仍不需要梯度。若漏掉`requires_grad_(False)`，這個輸出會變True；不能用包在`no_grad`裡得到的False替代凍結核對。實際DPO開始時通常複製已完成SFT的模型，整個偏好階段保留參考不變，並且不把它的參數放入policy優化器。

固定尺不表示參考所有回答都好。它用來控制新模型相對原行為的改變範圍，偏好資料仍需指出哪些方向更好；錯誤參考可以被新資料糾正。

參考在訓練起點與policy相同，故每個偏好對的相對差距初始為0，即使它們本來更常生成rejected也一樣。這讓目標關注相對偏好的改變，而不是要求兩個原分數本來已經一樣。更新過程中參考只負責再算同題兩篇答案的固定基準，不參與後面的梯度步。若每一步重新複製policy當reference，相對差距會不斷被歸零，失去一開始設定的固定約束意義。代碼裡的before是矩陣快照，clone複製數字；它和deepcopy整模型層級不同，卻都在避免引用同一份可變內容造成假核對。

練習只把那一行的`copy.deepcopy(policy)`替換成`policy`，保留後面的`.eval().requires_grad_(False)`，因此整行成為`reference = policy.eval().requires_grad_(False)`。先預測凍結會連policy也凍結，人為加0.1後第一行變False，再重跑核對，因為兩個名稱其實指同一份矩陣。這個反例說明複製、凍結與不收錄優化器是不同但相配的檢查。
